# 03 - Proposed method: Fisher split for acoustic TB labels

A TB label comes from sputum/Xpert/culture - it says the person has TB, not that
*this cough carries an audible sign of it*. Pooling three corpora makes the
mismatch worse: TBscreen is 72% TB, CIDRZ 17%, and each was recorded on its own
devices. A classifier trained on the raw labels spends its capacity on the
subset of TB+ clips that sound like nothing in particular.

The proposed method asks, for every TB+ clip, whether it actually sits on the
TB side of the acoustic space, and splits TB+ into two groups:

```
  embeddings ─► normalize ─► PCA ─► remove confounders (db, gender)
                                          │
                                          ▼
                              Fisher LDA  ──►  1-D axis
                                          │
                                          ▼
                          P(NTB | x) by Bayes + KDE per class
                                          │
                                          ▼
                       GMM(1) vs GMM(2) on the TB+ posteriors
                                          │
                        ┌─────────────────┴─────────────────┐
                 natural split                        no split
                        │                                   │
      TB+ splits into  ├── biomarker  (low P(NTB|x))    all TB+ kept
                       └── label-only (high P(NTB|x))    as a gradient
```

`disease_status_rev` then holds **0** = NTB, **1** = biomarker TB, **2** =
label-only TB, and the evaluation drops class 2 and re-scores.

Everything is inline in this notebook - nothing here imports from `lib/`, which
only covers building the embedding caches (notebooks 01 and 02).

**What this is not:** the split is fitted on the same labels it is judging, so a
better score after cleaning is not proof the discarded clips were mislabelled -
removing any hard subset raises accuracy. §7 is the honest part: it checks the
split against cleanlab, against a random drop of the same size, and against the
clinical metadata the split never saw.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (average_precision_score, balanced_accuracy_score,
                             confusion_matrix, roc_auc_score)
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.neighbors import KernelDensity
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
RANDOM_STATE = 42

## 0. Configuration

One switchboard for the whole notebook. `EMB_NAME` picks which cache from `02`
to run on; rerun the notebook end to end to compare embedders.

In [ ]:
EMB_NAME     = "qwen"      # "qwen" | "hear"
NORMALIZE    = "none"      # "none" | "l2" | "zscore" (per db)
CONFOUNDER   = "regression"  # "none" | "regression" | "combat"
N_COMPONENTS = 256         # PCA dims the split is computed in
PER_DB       = True        # fit the split inside each corpus separately
DO_REFINE    = False       # refit Fisher on biomarker TB+ only, then re-split

EXPERIMENT = (f"emb={EMB_NAME} norm={NORMALIZE} confound={CONFOUNDER} "
              f"pca={N_COMPONENTS} perdb={PER_DB} refine={DO_REFINE}")
print(EXPERIMENT)

`PER_DB` matters more than it looks. Pooled, the Fisher axis is free to
align with whatever separates the corpora - and since the corpora have wildly
different TB rates, "which corpus" is an excellent proxy for "TB". Fitting the
split inside each corpus removes that shortcut: the only contrast left is TB vs
NTB among clips recorded the same way.

## 1. Load

The `(metadata_{name}.csv, embeds_{name}.npy)` pair written by `02` is
row-aligned by construction. The assert is cheap and catches a stale cache.

In [ ]:
meta = pd.read_csv(f"result/metadata_{EMB_NAME}.csv")
embeddings_raw = np.load(f"result/embeds_{EMB_NAME}.npy")
assert len(meta) == len(embeddings_raw), "metadata and embeddings are out of sync"

labels     = meta["disease_status"].astype(int).values
gender     = meta["gender"].values
participant = meta["participant"].values
db         = meta["db"].values
db_name    = meta["db_name"].values
dbs        = sorted(meta["db_name"].unique())

print(f"{EMB_NAME}: {embeddings_raw.shape}")
print(meta.groupby(["db_name", "disease_status"]).size().unstack(fill_value=0))

## 2. Preprocessing

Three steps, each optional and each doing one thing.

**Normalize.** `l2` puts every clip on the unit sphere, which kills loudness /
gain differences but also any real intensity cue. `zscore` standardises within
each corpus - a crude batch correction on its own. `none` is the default because
the confounder step below is the deliberate version of the same idea.

**PCA.** Fisher LDA in 2048 dims with 45k clips will happily find a direction
that separates any labelling, including a random one. Cutting to 256 dims first
is the regularizer that keeps the axis meaningful.

**Remove confounders.** `regression` residualises every dimension against the
corpus one-hots and gender: fit a linear model predicting that dimension from
the confounds, keep the residual. `combat` (neuroCombat) instead harmonises the
per-corpus location and scale while protecting the disease effect, then
residualises gender.

In [ ]:
def normalize_embeddings(X, method, group=None):
    """Scale clips before the split. `group` is the per-corpus id for zscore."""
    if method == "none":
        return X.copy()
    if method == "l2":
        norms = np.clip(np.linalg.norm(X, axis=1, keepdims=True), 1e-10, None)
        return X / norms
    if method == "zscore":
        assert group is not None, "zscore needs a grouping"
        out = X.copy()
        for g in np.unique(group):
            m = group == g
            mu = out[m].mean(0)
            sd = np.clip(out[m].std(0), 1e-10, None)
            out[m] = (out[m] - mu) / sd
        return out
    raise ValueError(f"unknown normalization: {method}")


def fit_pca(X, n_components, seed=RANDOM_STATE):
    pca = PCA(n_components=n_components, random_state=seed)
    Z = pca.fit_transform(X)
    print(f"  PCA {X.shape[1]} -> {n_components} dims, "
          f"{pca.explained_variance_ratio_.sum() * 100:.1f}% variance kept")
    return pca, Z

In [ ]:
def build_confound_matrix(db_labels, gender_labels):
    """Corpus one-hots (drop-first) plus a gender indicator.

    Drop-first because the intercept already carries the reference corpus;
    keeping all of them makes the design matrix singular.
    """
    uniq = np.unique(db_labels)
    # uniq[1:] is empty when there is only one corpus (per-db mode, or a single
    # training fold) - column_stack of an empty list raises, so build it first.
    cols = [(db_labels == u).astype(float) for u in uniq[1:]]
    onehot = (np.column_stack(cols) if cols
              else np.zeros((len(db_labels), 0)))
    g = (gender_labels == np.unique(gender_labels)[0]).astype(float).reshape(-1, 1)
    return np.hstack([onehot, g])


def residualize(X, confounds):
    """Replace each dimension by what the confounds cannot explain."""
    if confounds.shape[1] == 0:
        return X.copy()
    lr = LinearRegression().fit(confounds, X)   # multi-output: one fit, all dims
    return X - lr.predict(confounds)


def apply_combat(X, db_labels, disease_labels):
    """neuroCombat: empirical-Bayes harmonisation of per-corpus mean/variance.

    `disease` is passed as a protected covariate so the harmonisation cannot
    flatten the very effect we are looking for.
    """
    from neuroCombat import neuroCombat
    covars = pd.DataFrame({"batch": db_labels, "disease": disease_labels})
    out = neuroCombat(dat=X.T, covars=covars, batch_col="batch",
                      categorical_cols=["disease"])
    return out["data"].T


def remove_confounders(X, method, db_labels, gender_labels, disease_labels=None):
    if method == "none":
        return X.copy()
    if method == "regression":
        return residualize(X, build_confound_matrix(db_labels, gender_labels))
    if method == "combat":
        assert disease_labels is not None
        harmonized = apply_combat(X, db_labels, disease_labels)
        g = (gender_labels == np.unique(gender_labels)[0]).astype(float).reshape(-1, 1)
        return residualize(harmonized, g)
    raise ValueError(f"unknown confounder method: {method}")

In [ ]:
normed = normalize_embeddings(embeddings_raw, NORMALIZE, group=db)
pca_model, reduced = fit_pca(normed, N_COMPONENTS)

if CONFOUNDER == "none":
    embeddings = reduced
else:
    embeddings = remove_confounders(reduced, CONFOUNDER, db, gender,
                                    disease_labels=labels)
print(f"split space: {embeddings.shape}")

### Did the confounder removal do anything?

A linear probe that can still name the corpus means the axis below can still use
it. Corpus accuracy should fall toward chance while TB AUROC survives - if TB
AUROC collapses too, the "TB signal" was largely corpus identity to begin with,
which is exactly the failure mode this notebook exists to expose.

In [ ]:
def verify_confounder_removal(X, db_labels, gender_labels, disease_labels,
                              cv=5, subsample=8000, seed=RANDOM_STATE):
    """Linear-probe readout of what is still recoverable from the embeddings."""
    rng = np.random.default_rng(seed)
    idx = (rng.choice(len(X), subsample, replace=False)
           if subsample and len(X) > subsample else np.arange(len(X)))
    probe = lambda y, scoring: cross_val_score(
        make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=1000, class_weight="balanced",
                                         random_state=seed)),
        X[idx], y[idx], cv=cv, scoring=scoring).mean()
    return {
        "db_acc": probe(db_labels, "accuracy"),
        "db_chance": float(pd.Series(db_labels[idx]).value_counts(normalize=True).max()),
        "gender_acc": probe(gender_labels, "accuracy"),
        "gender_chance": float(pd.Series(gender_labels[idx]).value_counts(normalize=True).max()),
        "tb_auroc": probe(disease_labels, "roc_auc"),
    }


for tag, X in [("before", reduced), ("after", embeddings)]:
    v = verify_confounder_removal(X, db, gender, labels)
    print(f"{tag:6s} db={v['db_acc']:.3f} (majority {v['db_chance']:.3f})  "
          f"gender={v['gender_acc']:.3f} (majority {v['gender_chance']:.3f})  "
          f"TB auroc={v['tb_auroc']:.3f}")

## 3. Fisher projection and P(NTB | x)

Fisher LDA collapses the 256-d space to the single direction that best separates
TB from NTB **as labelled**. Every clip gets a position on that axis; higher is
more TB-like (the sign is fixed after fitting so the direction never flips
between runs).

Then, per clip, Bayes' rule against the two class densities on that axis:

```
                P(x | NTB) · P(NTB)
P(NTB | x) = ───────────────────────────────────────────
             P(x | NTB) · P(NTB) + P(x | TB) · P(TB)
```

The likelihoods are Gaussian KDEs fitted to each labelled class, the priors are
the base rates. Everything stays in log-space - densities on a 45k-sample axis
underflow to zero if multiplied directly.

A TB+ clip with **high** P(NTB|x) is the interesting case: labelled TB, but
sitting where the NTB clips live.

In [ ]:
def silverman_bw(x):
    """Silverman's rule-of-thumb bandwidth for a 1-D KDE."""
    return max(1.09 * x.std() * len(x) ** (-0.2), 1e-3)


def fisher_projection(X, y, tb_mask_override=None):
    """1-D Fisher LDA axis. Returns (projection over ALL rows, direction).

    `tb_mask_override` narrows which TB+ rows define the positive class - used
    by the refinement pass, where the axis is refitted on biomarker TB+ only.
    Every row is still projected, including the ones left out of the fit.
    """
    if tb_mask_override is not None:
        fit_y = np.zeros(len(y))
        fit_y[tb_mask_override] = 1
        fit_mask = (y == 0) | tb_mask_override
    else:
        fit_y = y.copy()
        fit_mask = np.ones(len(y), dtype=bool)

    lda = LinearDiscriminantAnalysis(n_components=1)
    lda.fit(X[fit_mask], fit_y[fit_mask])
    proj = lda.transform(X).flatten()

    # LDA's sign is arbitrary; pin it so "higher = more TB-like" always holds.
    if proj[y == 1].mean() < proj[y == 0].mean():
        proj = -proj
    return proj, lda.coef_.flatten()


def posterior_ntb(proj, y):
    """P(NTB | position on the Fisher axis), computed in log-space."""
    tb_vals = proj[y == 1].reshape(-1, 1)
    ntb_vals = proj[y == 0].reshape(-1, 1)

    kde_tb = KernelDensity(bandwidth=silverman_bw(tb_vals)).fit(tb_vals)
    kde_ntb = KernelDensity(bandwidth=silverman_bw(ntb_vals)).fit(ntb_vals)

    log_prior_tb = np.log(max(y.mean(), 1e-10))
    log_prior_ntb = np.log(max(1 - y.mean(), 1e-10))

    pts = proj.reshape(-1, 1)
    log_tb = kde_tb.score_samples(pts) + log_prior_tb
    log_ntb = kde_ntb.score_samples(pts) + log_prior_ntb

    # log-sum-exp for the evidence term: subtract the larger value first so the
    # exponentials cannot overflow.
    log_max = np.maximum(log_tb, log_ntb)
    log_evidence = log_max + np.log(np.exp(log_tb - log_max) + np.exp(log_ntb - log_max))
    return np.exp(log_ntb - log_evidence)

## 4. The split

Take the P(NTB|x) values **of the TB+ clips only** and ask whether they form one
population or two. Fit GMMs with 1, 2 and 3 components and compare by BIC.

Declaring a split needs all three:

1. **BIC prefers 2 over 1** - there is more than one population.
2. **Separation ratio > 0.5** - the component means are at least half an average
   standard deviation apart, so the two are not the same blob split down the
   middle.
3. **The pair straddles 0.5, or the separation ratio exceeds 1.5** - one
   component is genuinely on the NTB side of the axis, or the two are far enough
   apart that it does not matter where they sit.

Without all three, the notebook reports a *gradient* instead: the posterior is
still a usable per-clip score, there is just no defensible place to cut it. That
is a real outcome, not a failure - forcing a boundary onto a unimodal
distribution is how this kind of method invents findings.

One thing the criterion deliberately ignores: BIC is compared 2-vs-1, but k=3 is
also fitted and printed. If `best_k` comes back 3 on every corpus, the posterior
is not cleanly two populations - the two-way cut is then a simplification chosen
for interpretability, and the split should be reported as such rather than as the
model the data preferred.

In [ ]:
def find_natural_split(post_ntb, y, verbose=True):
    """Is the TB+ posterior distribution one population or two?"""
    n = len(y)
    tb_idx = np.where(y == 1)[0]
    tb_post = post_ntb[tb_idx].reshape(-1, 1)

    gmms = {k: GaussianMixture(n_components=k, random_state=RANDOM_STATE,
                               n_init=10 if k > 1 else 1).fit(tb_post)
            for k in (1, 2, 3)}
    bic = {k: g.bic(tb_post) for k, g in gmms.items()}

    g2 = gmms[2]
    order = np.argsort(g2.means_.flatten())          # ascending mean posterior
    means = g2.means_.flatten()[order]
    stds = np.sqrt(g2.covariances_.flatten())[order]
    weights = g2.weights_[order]

    separation_ratio = abs(means[1] - means[0]) / max((stds[0] + stds[1]) / 2, 1e-10)
    straddles_05 = means[0] < 0.5 < means[1]
    has_split = (bic[1] - bic[2] > 0 and separation_ratio > 0.5
                 and (straddles_05 or separation_ratio > 1.5))

    res = {
        "tb_indices": tb_idx, "tb_posteriors": tb_post.flatten(),
        "bic": bic, "best_k": min(bic, key=bic.get),
        "comp_means": means.tolist(), "comp_stds": stds.tolist(),
        "comp_weights": weights.tolist(),
        "separation_ratio": separation_ratio, "straddles_05": straddles_05,
        "has_natural_split": has_split,
    }

    if verbose:
        print(f"  BIC 1/2/3 = {bic[1]:.1f} / {bic[2]:.1f} / {bic[3]:.1f} "
              f"(best k={res['best_k']})")
        print(f"  components mu={means.round(3).tolist()} "
              f"sd={stds.round(3).tolist()} w={weights.round(3).tolist()}")
        print(f"  separation={separation_ratio:.2f} straddles_0.5={straddles_05} "
              f"-> split={has_split}")

    lo_mask = np.zeros(n, dtype=bool)
    bio_mask = np.zeros(n, dtype=bool)

    if has_split:
        # `order` sorts components by mean posterior, so component 1 is the
        # NTB-leaning one: labelled TB, sitting where NTB clips sit.
        assign = np.argmax(g2.predict_proba(tb_post)[:, order], axis=1)
        lo_mask[tb_idx[assign == 1]] = True
        bio_mask[tb_idx[assign == 0]] = True
        res["component_assignment"] = assign
        res["label_only_fraction"] = float((assign == 1).mean())
        if verbose:
            print(f"  -> biomarker {int(bio_mask.sum())} | "
                  f"label-only {int(lo_mask.sum())} "
                  f"({res['label_only_fraction'] * 100:.1f}% of TB+)")
    else:
        bio_mask[tb_idx] = True                       # keep every TB+ clip
        pcts = np.percentile(tb_post, [10, 25, 50, 75, 90])
        res["gradient_percentiles"] = dict(zip(["p10", "p25", "p50", "p75", "p90"],
                                               pcts.round(3)))
        if verbose:
            print(f"  -> no natural split; posterior gradient "
                  f"{res['gradient_percentiles']}")

    res["label_only_mask"] = lo_mask
    res["biomarker_mask"] = bio_mask
    return res

### Refinement (optional)

If a split exists, the axis that produced it was fitted on *contaminated* TB+
labels - the label-only clips pulled the TB centroid toward NTB. Refitting Fisher
on the biomarker subset alone gives a cleaner axis, and the split is recomputed
on it.

It is off by default: the refit is fitted on its own previous output, so each
round makes the two groups look more separated whether or not that separation is
real. Turn it on to see how far the split moves, not to make it look better.

In [ ]:
def refine_with_split(X, y, split, verbose=True):
    """Refit the Fisher axis on biomarker TB+ only, then re-split on it."""
    if not split["has_natural_split"]:
        if verbose:
            print("  no split to refine")
        return split

    bio = split["biomarker_mask"]
    if verbose:
        print(f"  refitting Fisher on {int(bio.sum())} biomarker TB+ "
              f"(dropping {int(split['label_only_mask'].sum())} label-only)")

    proj, direction = fisher_projection(X, y, tb_mask_override=bio)
    post = posterior_ntb(proj, y)
    refined = find_natural_split(post, y, verbose=verbose)
    refined.update(projected_1d=proj, posterior_ntb=post,
                   fisher_direction=direction, is_refined=True)
    return refined


def fisher_split(X, y, do_refinement=DO_REFINE, verbose=True):
    """Fisher -> posterior -> split (-> refine) on one homogeneous subset."""
    proj, direction = fisher_projection(X, y)
    post = posterior_ntb(proj, y)
    split = find_natural_split(post, y, verbose=verbose)
    split.update(projected_1d=proj, posterior_ntb=post,
                 fisher_direction=direction, is_refined=False)

    if do_refinement and split["has_natural_split"]:
        refined = refine_with_split(X, y, split, verbose=verbose)
        # Only keep the refit if it still finds a split; a refinement that
        # dissolves its own split is evidence against the original one.
        if refined["has_natural_split"]:
            return refined
        if verbose:
            print("  refinement lost the split - keeping the original")
    return split

### Per-corpus wrapper

Runs `fisher_split` on each corpus separately and scatters the per-clip outputs
back into full-length arrays, so downstream code sees one `posterior_ntb` and one
`biomarker_mask` regardless of which mode was used.

Note the posteriors are only comparable *within* a corpus in this mode - each has
its own Fisher axis, its own KDEs and its own priors. Pooling them into one
histogram would be meaningless; the plots below facet by corpus for that reason.

In [ ]:
PER_SAMPLE_KEYS = ("biomarker_mask", "label_only_mask", "projected_1d",
                   "posterior_ntb")


def run_per_database(X, y, group, verbose=True, **kwargs):
    """Split each corpus on its own clips, then stitch the results together."""
    X, y, group = np.asarray(X), np.asarray(y), np.asarray(group)
    n = len(y)
    merged, per_db, any_split = {}, {}, False

    for g in np.unique(group):
        m = group == g
        if len(np.unique(y[m])) < 2:
            print(f"  skipping {g!r}: only one class present")
            per_db[g] = {"skipped": True}
            continue
        if verbose:
            print(f"\n=== {g} (n={int(m.sum())}, TB+={int((y[m] == 1).sum())}) ===")

        sub = fisher_split(X[m], y[m], verbose=verbose, **kwargs)
        per_db[g] = sub
        any_split |= bool(sub["has_natural_split"])

        for key in PER_SAMPLE_KEYS:
            val = sub.get(key)
            if not isinstance(val, np.ndarray) or val.shape[0] != m.sum():
                continue
            if key not in merged:
                merged[key] = (np.zeros(n, dtype=bool) if val.dtype == bool
                               else np.full(n, np.nan))
            merged[key][m] = val

    merged.setdefault("label_only_mask", np.zeros(n, dtype=bool))
    merged.setdefault("biomarker_mask", y == 1)
    merged.update(per_database_results=per_db, per_database=True,
                  has_natural_split=any_split,
                  tb_indices=np.where(y == 1)[0])
    return merged


def run_split(X, y, group=None, verbose=True, **kwargs):
    """Entry point: per-corpus when `group` is given, pooled otherwise."""
    if group is not None:
        return run_per_database(X, y, group, verbose=verbose, **kwargs)
    return fisher_split(X, y, verbose=verbose, **kwargs)

## 5. Run it

`disease_status_rev` is the output of this whole notebook:

| value | meaning |
|---|---|
| 0 | NTB (untouched) |
| 1 | biomarker TB - labelled TB **and** acoustically TB-like |
| 2 | label-only TB - labelled TB, sits with the NTB clips |

In [ ]:
split = run_split(embeddings, labels,
                  group=db_name if PER_DB else None,
                  do_refinement=DO_REFINE, verbose=True)

df = meta.copy()
df["fisher_projection"] = split["projected_1d"]
df["posterior_ntb"] = split["posterior_ntb"]
df["acoustic_tb_score"] = 1.0 - split["posterior_ntb"]

tb_mask = df["disease_status"] == 1
df["disease_status_rev"] = 0
df.loc[tb_mask & split["biomarker_mask"], "disease_status_rev"] = 1
df.loc[tb_mask & split["label_only_mask"], "disease_status_rev"] = 2

print(f"\nTB+ clips: {int(tb_mask.sum())}")
print(df.loc[tb_mask, "disease_status_rev"]
        .map({1: "biomarker", 2: "label-only"}).value_counts())

In [ ]:
# Where the label-only clips come from, by corpus and by subject.
per_db = (df[tb_mask]
          .groupby(["db_name", "disease_status_rev"]).size().unstack(fill_value=0)
          .rename(columns={1: "biomarker", 2: "label_only"}))
per_db["label_only_pct"] = (per_db.get("label_only", 0) /
                            per_db.sum(axis=1) * 100).round(1)
print(per_db, "\n")

# A split that fires on whole subjects is a different claim from one that fires
# on scattered clips: the first says "this patient's cough carries nothing",
# the second says "this recording didn't catch it".
subj = (df[tb_mask].groupby("participant")["disease_status_rev"]
        .agg(n="size", label_only=lambda s: (s == 2).mean()))
print(f"TB+ subjects: {len(subj)}")
print(f"  all clips label-only : {(subj.label_only == 1).sum()}")
print(f"  all clips biomarker  : {(subj.label_only == 0).sum()}")
print(f"  mixed                : {((subj.label_only > 0) & (subj.label_only < 1)).sum()}")

## 6. What the split looks like

Left column: where TB+ and NTB clips land on each corpus's Fisher axis. Right
column: the P(NTB|x) distribution of the TB+ clips only, which is the thing the
GMM is fitted to - a split is credible when this panel is visibly bimodal, and
suspicious when the notebook reports a split on a panel that looks like one
lump.

In [ ]:
fig, axes = plt.subplots(len(dbs), 2, figsize=(13, 3.4 * len(dbs)), squeeze=False)

for row, name in enumerate(dbs):
    sel = df["db_name"] == name
    ax = axes[row][0]
    for lab, colour, tag in [(0, "#4E4CAF", "NTB"), (1, "#F3213A", "TB")]:
        vals = df.loc[sel & (df["disease_status"] == lab), "fisher_projection"]
        if len(vals) > 1:
            sns.kdeplot(vals, fill=True, alpha=0.4, color=colour,
                        label=f"{tag} (n={len(vals)})", ax=ax)
    ax.set(title=f"{name} - Fisher axis", xlabel="projection (higher = TB-like)")
    ax.legend(frameon=False)

    ax = axes[row][1]
    tb_post = df.loc[sel & (df["disease_status"] == 1), "posterior_ntb"]
    bio = df.loc[sel & (df["disease_status_rev"] == 1), "posterior_ntb"]
    lo = df.loc[sel & (df["disease_status_rev"] == 2), "posterior_ntb"]
    ax.hist(tb_post, bins=60, color="#BBBBBB", label="all TB+")
    if len(lo):
        ax.hist([bio, lo], bins=60, stacked=True, color=["#F3213A", "#F5A623"],
                label=None)
        ax.hist([], color="#F3213A", label="biomarker")
        ax.hist([], color="#F5A623", label="label-only")
    sub = split.get("per_database_results", {}).get(name, split)
    if sub.get("has_natural_split"):
        for mu in sub["comp_means"]:
            ax.axvline(mu, ls="--", lw=1, color="k")
    ax.set(title=f"{name} - P(NTB|x) of TB+ clips", xlabel="P(NTB | x)")
    ax.legend(frameon=False)

plt.tight_layout()
plt.show()

## 7. Does the split mean anything?

The split was fitted on the same labels it is judging, so "accuracy went up" is
not evidence - dropping any hard subset raises accuracy. Four checks, in
increasing order of how much they can hurt the claim:

1. **Classification, cleaned vs raw** - the headline number, and the weakest.
   Always paired with the random-drop control below.
2. **Random-drop control** - discard the same number of random TB+ clips.
   Whatever this gains is the part of the improvement that is just an easier
   task, not better labels.
3. **Agreement with cleanlab** - an independent noise detector on the same
   embeddings. High overlap means two different methods flag the same clips.
4. **Clinical read-out** - the split never saw smear/culture/Xpert grade,
   symptoms or TB type. If label-only clips skew toward clinically-diagnosed
   rather than bacteriologically-confirmed TB, that is external support the
   other three cannot give.

All evaluation runs on the **raw** embeddings, not the confounder-removed ones:
the cleaning decided *which rows to keep*, and scoring on the space it was fitted
in would bake its own transform into the result.

In [ ]:
def fold_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "bacc": balanced_accuracy_score(y_true, y_pred),
        "sens": tp / max(tp + fn, 1),
        "spec": tn / max(tn + fp, 1),
        "auroc": roc_auc_score(y_true, y_prob),
        "auprc": average_precision_score(y_true, y_prob),
    }


def grouped_cv(X, y, groups, X_ext=None, y_ext=None, n_splits=5,
               n_components=256, seed=RANDOM_STATE):
    """Subject-grouped CV. Scaler+PCA are fitted per fold, inside the training
    split only - fitting them on everything first leaks test rows into the
    projection and inflates every metric downstream.

    `X_ext`/`y_ext`: an external corpus scored by each fold's model (LODO).
    """
    n_components = min(n_components, X.shape[1], len(y) // 2)
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    rows, rows_ext = [], []

    for tr, te in cv.split(X, y, groups):
        pipe = make_pipeline(
            StandardScaler(),
            PCA(n_components=n_components, random_state=seed),
            LogisticRegression(max_iter=2000, class_weight="balanced",
                               random_state=seed),
        ).fit(X[tr], y[tr])
        rows.append(fold_metrics(y[te], pipe.predict_proba(X[te])[:, 1]))
        if X_ext is not None:
            rows_ext.append(fold_metrics(y_ext, pipe.predict_proba(X_ext)[:, 1]))

    out = pd.DataFrame(rows).mean().to_dict()
    out["n"] = len(y)
    out["tb_rate"] = float(y.mean())
    if rows_ext:
        for k, v in pd.DataFrame(rows_ext).mean().items():
            out[f"ext_{k}"] = v
    return out

In [ ]:
def evaluate(keep_mask, stage, n_splits=5):
    """Within-corpus CV for every corpus, on the rows `keep_mask` selects."""
    rows = []
    for name in dbs:
        m = keep_mask & (df["db_name"] == name).values
        y = df.loc[m, "disease_status"].astype(int).values
        if len(np.unique(y)) < 2:
            continue
        rows.append({"stage": stage, "db": name,
                     **grouped_cv(embeddings_raw[m], y,
                                  df.loc[m, "participant"].values,
                                  n_splits=n_splits)})
    return pd.DataFrame(rows)


raw_mask = np.ones(len(df), dtype=bool)
clean_mask = (df["disease_status_rev"] != 2).values

results = pd.concat([evaluate(raw_mask, "raw"),
                     evaluate(clean_mask, "cleaned")], ignore_index=True)
print(results.round(4).to_string(index=False))

In [ ]:
# Control: drop the same number of TB+ clips per corpus, chosen at random.
# The gap between "cleaned" and "random-drop" is the part of the improvement
# that the split earned; the rest is just a smaller, easier problem.
rng = np.random.default_rng(RANDOM_STATE)
random_mask = np.ones(len(df), dtype=bool)
for name in dbs:
    sel = (df["db_name"] == name).values
    n_drop = int((clean_mask == False)[sel].sum())
    tb_idx = np.where(sel & (df["disease_status"] == 1).values)[0]
    if n_drop and len(tb_idx) > n_drop:
        random_mask[rng.choice(tb_idx, n_drop, replace=False)] = False

control = evaluate(random_mask, "random-drop")
comparison = (pd.concat([results, control], ignore_index=True)
              .pivot(index="db", columns="stage",
                     values=["auroc", "bacc", "auprc"]).round(4))
print(comparison.to_string())

In [ ]:
# Leave-one-corpus-out: train on two corpora, test on the third. This is the
# number that says whether the cleaning generalises or just fits each corpus.
lodo = []
for stage, mask in [("raw", raw_mask), ("cleaned", clean_mask),
                    ("random-drop", random_mask)]:
    for held_out in dbs:
        tr = mask & (df["db_name"] != held_out).values
        te = mask & (df["db_name"] == held_out).values
        y_tr = df.loc[tr, "disease_status"].astype(int).values
        y_te = df.loc[te, "disease_status"].astype(int).values
        if len(np.unique(y_tr)) < 2 or len(np.unique(y_te)) < 2:
            continue
        r = grouped_cv(embeddings_raw[tr], y_tr, df.loc[tr, "participant"].values,
                       X_ext=embeddings_raw[te], y_ext=y_te)
        lodo.append({"stage": stage, "held_out": held_out,
                     "auroc": r["ext_auroc"], "bacc": r["ext_bacc"],
                     "sens": r["ext_sens"], "spec": r["ext_spec"]})

lodo_df = pd.DataFrame(lodo)
print(lodo_df.pivot(index="held_out", columns="stage",
                    values=["auroc", "bacc"]).round(4).to_string())

### 7.3 Agreement with cleanlab

cleanlab finds label issues a different way: cross-validated predicted
probabilities, then the confident-joint estimate of the noise rate. It shares the
embeddings with the proposed method and nothing else - no Fisher axis, no KDE, no
GMM.

Two things to read here. Overlap well above what chance would give means the two
methods are pointing at the same clips. But cleanlab flags issues in **both**
classes, while the Fisher split only ever reassigns TB+, so the comparison is
restricted to TB+ rows.

In [ ]:
from cleanlab.count import estimate_cv_predicted_probabilities
from cleanlab.filter import find_label_issues


def run_cleanlab(X, y, groups=None, n_splits=5, seed=RANDOM_STATE):
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight="balanced", random_state=seed),
    )
    proba = estimate_cv_predicted_probabilities(X, y, clf=clf, cv_n_folds=n_splits,
                                               seed=seed)
    issues = find_label_issues(y, proba, filter_by="prune_by_noise_rate")
    return issues, proba


cl_issue = np.zeros(len(df), dtype=bool)
for name in dbs:                    # per corpus, to match PER_DB
    m = (df["db_name"] == name).values
    y_sub = df.loc[m, "disease_status"].astype(int).values
    issues, _ = run_cleanlab(embeddings[m], y_sub)
    cl_issue[np.where(m)[0][issues]] = True

df["cleanlab_issue"] = cl_issue

tb = tb_mask.values
fisher_flag = (df["disease_status_rev"] == 2).values
both = (fisher_flag & cl_issue & tb).sum()
# Chance overlap if the two flagged independently at their observed rates.
expected = (fisher_flag & tb).sum() * (cl_issue & tb).sum() / max(tb.sum(), 1)
jaccard = both / max(((fisher_flag | cl_issue) & tb).sum(), 1)

print(f"TB+ clips flagged by Fisher split : {(fisher_flag & tb).sum()}")
print(f"TB+ clips flagged by cleanlab     : {(cl_issue & tb).sum()}")
print(f"flagged by both                   : {both}  "
      f"(chance would give ~{expected:.0f})")
print(f"Jaccard over TB+                  : {jaccard:.3f}")

### 7.4 Clinical read-out

The strongest check available, because none of these columns entered the split -
not as a feature, not as a confound, not as a label.

**Bacterial load.** A smear-negative or high-Ct patient is paucibacillary: less
disease in the airway, so less reason for the cough to sound different. If the
label-only group is where those patients concentrate, the split is tracking
disease burden rather than noise.

**Symptoms.** Cough character follows symptoms. A label-only group with a
systematically lighter symptom burden is consistent with the clips genuinely
carrying less TB signal.

Small differences here are expected and weak. A *reversal* - label-only clips
being the sicker ones - would be a real problem for the method.

In [ ]:
# Pull the lab columns straight from 01's output. They are joined on path_file,
# which is unique per clip, and none of them exist in the unified metadata the
# split ran on.
cidrz_lab = (pd.read_csv("data/cidrz.csv", low_memory=False)
             [["path_file", "smear_all", "xpert_results", "culture_results"]])
tbs_lab = pd.read_csv("data/tbscreen.csv", low_memory=False)[
    ["path_file", "lab_sptm_xpert_ct_a", "lab_sptm_smr_rslt_a"]]
tbs_lab["xpert_ct"] = pd.to_numeric(tbs_lab["lab_sptm_xpert_ct_a"], errors="coerce")
tbs_lab["smear_grade"] = pd.to_numeric(tbs_lab["lab_sptm_smr_rslt_a"], errors="coerce")

clin = (df.merge(cidrz_lab, on="path_file", how="left")
          .merge(tbs_lab[["path_file", "xpert_ct", "smear_grade"]],
                 on="path_file", how="left"))
group_name = clin["disease_status_rev"].map({1: "biomarker", 2: "label-only"})

# CIDRZ: smear positivity among TB+ clips.
cid = clin[(clin["db_name"] == "cidrz") & (clin["disease_status"] == 1)]
if len(cid) and cid["disease_status_rev"].nunique() > 1:
    print("CIDRZ TB+ - smear-positive rate (higher = more bacilli):")
    print(cid.groupby(group_name[cid.index])["smear_all"]
             .agg(n="size", smear_pos_rate="mean").round(3), "\n")

# TBscreen: Xpert Ct among TB+ clips (LOWER Ct = MORE bacilli).
tbs = clin[(clin["db_name"] == "tbscreen") & (clin["disease_status"] == 1)]
tbs = tbs[tbs["xpert_ct"].notna()]
if len(tbs) and tbs["disease_status_rev"].nunique() > 1:
    print("TBscreen TB+ - Xpert Ct (lower = higher bacterial load):")
    print(tbs.groupby(group_name[tbs.index])["xpert_ct"]
             .agg(n="size", mean_ct="mean", median_ct="median").round(2), "\n")
    from scipy import stats
    a = tbs.loc[tbs["disease_status_rev"] == 1, "xpert_ct"]
    b = tbs.loc[tbs["disease_status_rev"] == 2, "xpert_ct"]
    if len(a) > 1 and len(b) > 1:
        u, p = stats.mannwhitneyu(a, b)
        print(f"  Mann-Whitney biomarker vs label-only: p={p:.3g}\n")

In [ ]:
# Symptom burden among TB+ clips. These five columns are in the unified metadata
# but were never given to the split.
SYMPTOMS = ["weight_loss", "hemoptysis", "night_sweats", "fever", "smoker"]
sym = df[tb_mask].copy()
sym[SYMPTOMS] = sym[SYMPTOMS].replace(-1, np.nan)   # -1 is "unknown", not "no"
sym["n_symptoms"] = sym[["weight_loss", "hemoptysis", "night_sweats", "fever"]].sum(
    axis=1, min_count=1)

table = (sym.groupby(["db_name", sym["disease_status_rev"]
                      .map({1: "biomarker", 2: "label-only"})])
            [SYMPTOMS + ["n_symptoms"]].mean().round(3))
print("mean symptom positivity among TB+ clips:")
print(table.to_string())

### 7.5 The nested check - the only number that tests the claim

§7.1 fits the split on **every** clip and then cross-validates on those same
clips. The split's whole job is to remove the TB+ clips a linear model cannot
separate from NTB, and the evaluation is a linear model on the same embeddings,
so it removes exactly the rows that would have been scored wrong. AUROC has to
go up. The random-drop control in §7.2 does not rescue this - random clips are
not the hard ones, so it only shows the gain is not from having fewer rows.

The claim a label-cleaning method actually makes is narrower: **cleaning the
training labels produces a better model**. That is testable without circularity:

```
for each fold:
    fit preprocessing + Fisher split on TRAIN rows only
    drop the label-only TRAIN rows
    train the classifier on what is left
    score the TEST fold with its ORIGINAL labels, nothing removed
```

The test fold is never cleaned and never seen by the split, so the three stages
are scored on identical rows against identical labels. If cleaning helps, the
`cleaned` column beats both others here. If it does not, §7.1 was measuring
itself.

In [ ]:
def nested_clean_cv(name, n_splits=5, seed=RANDOM_STATE):
    """Fit the cleaner inside each training fold; score the untouched test fold."""
    sel = (df["db_name"] == name).values
    X_raw = embeddings_raw[sel]
    y = df.loc[sel, "disease_status"].astype(int).values
    groups = df.loc[sel, "participant"].values
    g_gender, g_db = gender[sel], db[sel]

    rng = np.random.default_rng(seed)
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    rows = []

    for fold, (tr, te) in enumerate(cv.split(X_raw, y, groups)):
        # --- the cleaner, fitted on training rows only ---
        normed_tr = normalize_embeddings(X_raw[tr], NORMALIZE, group=g_db[tr])
        pca_tr = PCA(n_components=N_COMPONENTS, random_state=seed).fit(normed_tr)
        Z_tr = remove_confounders(pca_tr.transform(normed_tr), CONFOUNDER,
                                  g_db[tr], g_gender[tr], disease_labels=y[tr])
        sp = fisher_split(Z_tr, y[tr], verbose=False)

        keep = ~sp["label_only_mask"]
        n_drop = int((~keep).sum())

        # Same-size random drop from the training TB+ rows, same folds.
        rand_keep = np.ones(len(tr), dtype=bool)
        tb_local = np.where(y[tr] == 1)[0]
        if n_drop and len(tb_local) > n_drop:
            rand_keep[rng.choice(tb_local, n_drop, replace=False)] = False

        for stage, mask in [("raw", np.ones(len(tr), dtype=bool)),
                            ("cleaned", keep), ("random-drop", rand_keep)]:
            y_fit = y[tr][mask]
            if len(np.unique(y_fit)) < 2:
                continue
            pipe = make_pipeline(
                StandardScaler(),
                PCA(n_components=min(N_COMPONENTS, X_raw.shape[1]), random_state=seed),
                LogisticRegression(max_iter=2000, class_weight="balanced",
                                   random_state=seed),
            ).fit(X_raw[tr][mask], y_fit)
            # Test fold: original labels, every row kept.
            rows.append({"db": name, "fold": fold, "stage": stage,
                         "n_train": int(mask.sum()), "n_dropped": int((~mask).sum()),
                         **fold_metrics(y[te], pipe.predict_proba(X_raw[te])[:, 1])})
    return pd.DataFrame(rows)


nested = pd.concat([nested_clean_cv(name) for name in dbs], ignore_index=True)
print(nested.groupby(["db", "stage"])[["auroc", "bacc", "sens", "spec"]]
            .mean().round(4).to_string())

In [ ]:
# Fold-paired difference: cleaned minus raw, within each fold. Averaging the
# per-fold gap is stricter than comparing two averages - the folds share the
# same test rows, so a positive mean with a consistent sign is the signal.
wide = nested.pivot_table(index=["db", "fold"], columns="stage", values="auroc")
delta = pd.DataFrame({
    "cleaned - raw": wide["cleaned"] - wide["raw"],
    "cleaned - random": wide["cleaned"] - wide["random-drop"],
    "random - raw": wide["random-drop"] - wide["raw"],
})
print("per-fold AUROC differences (mean over folds, [min, max]):")
for db_name_ in dbs:
    d = delta.loc[db_name_]
    print(f"  {db_name_:9s} " + "  ".join(
        f"{c}={d[c].mean():+.4f} [{d[c].min():+.4f}, {d[c].max():+.4f}]"
        for c in delta.columns))
print("\nnested drop counts per fold:")
print(nested[nested.stage == "cleaned"]
      .groupby("db")[["n_train", "n_dropped"]].mean().round(0).to_string())

## 8. Save

Writes the per-clip verdict next to the embedding caches. `disease_status` is
overwritten with `disease_status_rev` so downstream training code can consume it
without knowing this notebook exists; the diagnostic columns ride along under
their own names.

In [ ]:
out = df.drop(columns=["disease_status_rev"]).copy()
out["disease_status"] = df["disease_status_rev"]
out["disease_status_orig"] = df["disease_status"]

out_path = f"result/metadata_cleaned_{EMB_NAME}.csv"
out.to_csv(out_path, index=False)
print(f"saved {out_path}  ({len(out)} rows)")
print(out["disease_status"].value_counts().rename(
    {0: "NTB", 1: "biomarker TB", 2: "label-only TB"}))

results.assign(experiment=EXPERIMENT).to_csv(
    f"result/eval_perdb_{EMB_NAME}.csv", index=False)
lodo_df.assign(experiment=EXPERIMENT).to_csv(
    f"result/eval_lodo_{EMB_NAME}.csv", index=False)
print(f"\n{EXPERIMENT}")

## 9. Reading the result

The three evaluations answer different questions and can disagree. In order of
what they are worth:

| section | question | can it be gamed by the split? |
|---|---|---|
| 7.1 in-sample | do the kept clips classify better? | **yes, by construction** |
| 7.2 random-drop | is the gain just from fewer rows? | no, but it does not test the labels |
| 7.3 cleanlab | does an independent method flag the same clips? | no |
| 7.4 clinical | do label-only clips have less disease? | no - unseen columns |
| **7.5 nested** | **does cleaning the training labels give a better model?** | **no** |

§7.1 will always look spectacular. The split removes the TB+ clips a linear probe
cannot separate from NTB, and §7.1 scores a linear probe on those same rows -
the improvement is the method grading its own homework. Report it only next to
§7.5.

**§7.5 is the result.** A method that cleans labels has to produce a model that
does better on data it never touched. If `cleaned - raw` is negative or sits
inside the fold-to-fold spread, the split is not finding mislabelled clips; it is
finding hard ones, and discarding them costs the classifier the examples it most
needed. That outcome is publishable - "this posterior separates hard clips from
easy ones, not true labels from false ones" is a finding - but it is not the
same claim as label cleaning, and the abstract has to say which one it is.

To probe further from here: rerun with `EMB_NAME="hear"` (does the split survive
a different encoder?), with `PER_DB=False` (how much of it was corpus identity?),
and with `DO_REFINE=True` (does iterating make the split sharper, or just make it
eat more of the TB+ class?).